---
title: "Calibration methods"
---

## Post-hoc calibration for binary classification

Most calibration methods for ML models are post-hoc, i.e. the model is first trained without calibration in mind and only after that the model output is calibrated to try to align the model confidence with the true posterior probability. Defining "true posterior probability" properly may be difficult, but for the purpose of this section we shall adopt the definition that true posterior probability is the frequency found in a representative test set. I.e., if we select instances classified to have a confidence in a narrow range around $\alpha$, then the fraction of correctly classified events should be $\alpha$ if the model is properly calibrated.

Schematically, the goal of the calibration is to move from the plot on the left to the plot on the right. On the left is a situation which seems to be typical for the complex neural networks that have become popular recently. The confidence is boosted such that allmost all of the correctly classified instances come with a very high confidence. If one measures the average confidence this comes out as (much larger) than the accuracy. The plot to the right shows an ideal case, where there is an excellent correspondance between confidence and accuracy.

![](./images/calibration.png)

The presention of calibration methods below is based on [Guo2017]^[[On Calibration of Modern Neural Networks](https://proceedings.mlr.press/v70/guo17a.html) \
C Guo, G Pleiss, Y Sun, and K Weinberger \
Proceedings of the 34th International Conference on Machine Learning, PMLR 70:1321-1330, 2017] where also more details can be found.

### Histogram binning
Define bin boundaries $0=b_1\leq b_2 \leq \ldots \leq b_{M+1} = 1$ such that classified^[We assume that the output of the ML model is normalized to be in the range $[0,1]$, for instance by using softmax.] instances are put in mutually exclusive bins. There is no hard rule on how to choose the bin boundaries, but typically one would either choose uniform bin width or (approximately) equal number of instances per bin. For each bin, the calibrated confidence is determined to be the value $\theta_m$ that minimizes the squared loss of the instances in the bin:
$$\min_{\theta_m} \sum_{i=1}^{n}(\theta_m - y_i)^2$$
where $y_i \in \{0,1\}$ is the target value for instance $i$. 

### Isotonic regression
This is a generalisation of the histogram binning approach where also the binning is subject to opitmization. In specific, let the calibrated confidences per bin $\theta_m$ be determined by minimising
$$
\min_{\stackrel{M}{\stackrel{\theta_1,\ldots\theta_M}{b_1\ldots b_{M+1}}}}
\sum_{m=1}{M}\sum_{i=1}^{N}{\mathbf I}(b_i\leq \hat{p}_i < b_{i+1}) (\theta_m-y_i)^2
$$
subject to $0=b_1\leq b_2 \leq \ldots \leq b_{M+1} = 1$ and $\theta_1 \leq\theta_2\leq\ldots\theta_M$. Here $\hat{p_i}$ is the ML model's (pre-calibration) confidence of instance $p_i$, and ${\mathbf I}(\cdot)$ is the indicator function. Thus each instance contribute to the sum, only in the bin where it belongs. Note that in this approach both number of bins as well as the different bin widths are determined through the minimisation procedure.

### Bayesian Binning into Quantiles (BBQ)
This is another generalisation of histogram binning which is based on Bayesian model averaging. Instead of, like isotonic regression, just letting the selected binning be subject to optimisation BBQ considers all possible binnings that respect the constraint $0=b_1\leq b_2 \leq \ldots \leq b_{M+1} = 1$ simultaneously and averages over the probabilities given by each binning. The weights used in the averaging are found using Bayes' rule.

### Platt scaling
This is a standard linear regression applied to the predictions of the pre-calibrated ML model. Let $z_i$ be the confidence before calibration. Post-calibration confidences are then calculated as
$$\hat{q}_i = \sigma(az_i + b)$$
where $\sigma(\cdot)$ is the standard sigmoid function and the parameters $a$ and $b$ are optimized using the validation set.


## Challenges when extending to multiclass calibration

In a multiclass setting, the ML model outputs one confidence per class per instance, i.e. the prediction for an instance may be
$$ [0.73, 0.06, 0.13, 0.08]$$
signinfying that the model holds highest confidence to placing the instance in the first class, but the other classes are also possible. 
What do we expect from the calibration? Should only the confidence of the most probable class be "correct"? Or should we require all classes to have "correct" confidence? Multiple definitions of calibration are possible. Following [Perez-Lebel2023]^[[Beyond calibration: estimating the grouping loss of modern neural networks](https://arxiv.org/abs/2210.16315) \ A Perez-Lebel, M Le Morvan, and G Varoquaux \ ICLR 2023] we consider three different definitions. In all the definitions, let $s = (s_1,\ldots, s_k)$ be the per-class confidence scores and  $\{e_1, \ldots ,e_K\}$ be the set of all one-hot vectors of size K^[I.e. $e_m=(0,\ldots,0,1,0,\ldots 0)$ where the 1 is vector element number $m$.].

With these definitions $P(Y=e_k|S=s)$ should be read as the probability for the true value of the instance being class $k$ given that the ML model produces the confidence score vector $s$. 

**Joint calibration**^[Check that the interpretation of the three definitions actually are correct. Particularly joint and class-wise calibration is somewhat confusing.] is the strongest of the three definitions and requires 
$$P(Y=e_k|S = s) = s_k \quad\text{for}\quad k=1,\ldots,K$$
i.e., we require that all of the confusion matrix is properly calibrated.

A slightly weaker definition is **class-wise calibration** where we require
$$P(Y=e_k|S_k = s_k) = s_k \quad\text{for}\quad k=1,\ldots,K.$$
i.e., we no longer require that all of the confusion matrix is properly calibrated, but only the confidences for the class the element actually belongs to.

An even weaker definition is **top-label calibration** where we require
$$P(Y=e_{\mathrm{argmax}(s)}|\max(S) = s) = s \quad\text{for}\quad k=1,\ldots,K$$
i.e., we require only that the probability for the class with highest confidence is correct.
